# AQUAGUARD AI – GROUNDWATER RISK & WATER QUALITY CLASSIFICATION
### Member 2: Sakshi Pawar | Third Year AIML | Academic Project
**Project Title**: *AI-Based Groundwater Level Prediction and Geographical Risk Assessment Using Machine Learning*

**Assigned Module**: Part 2 – Groundwater Resource Risk Classification and Water Quality Potability Classification

---

## 1. Setup & Library Imports
We import data manipulation, visualization, preprocessing, and machine learning libraries.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries successfully imported!')

## 2. Dataset Loading
Loading Maharashtra Groundwater Resource Assessment (2025) and Water Quality Chemical Parameters (1961-2025).

In [ ]:
# In Google Colab, upload the CSV files or adjust paths
res_path = 'mh_gw_2025.csv' if os.path.exists('mh_gw_2025.csv') else 'data/raw/mh_gw_2025.csv'
wq_path = 'gwq_chemical_parameter_manual_cgwb_an_1961_2025.csv' if os.path.exists('gwq_chemical_parameter_manual_cgwb_an_1961_2025.csv') else 'data/raw/gwq_chemical_parameter_manual_cgwb_an_1961_2025.csv'

df_res_raw = pd.read_csv(res_path)
df_wq_raw = pd.read_csv(wq_path)

# Clean Resource dataset (exclude summary/total rows)
df_res = df_res_raw[df_res_raw['Name of District'].notnull()].copy()
df_res = df_res[~df_res['Name of District'].astype(str).str.contains('Total', case=False, na=False)].reset_index(drop=True)

print(f'Resource Assessment Districts: {len(df_res)}')
print(f'Water Quality Records: {len(df_wq_raw)}')

## 3. Data Preprocessing & Target Variable Preparation
- **Resource Risk (CGWB Rules)**: Safe (<=70%), Semi-Critical (70-90%).
- **Target Leakage Prevention**: We strictly DROP `Stage of GW extraction (%)` from feature predictors.
- **Water Quality (BIS IS 10500:2012)**: Binary Potability labeled via standard drinking water permissible limits.

In [ ]:
# 1. Resource Target Formulation
stage_col = 'Stage of GW extraction (%)'
df_res['Target'] = (df_res[stage_col] > 70.0).astype(int)
drop_cols = ['Sr. No.', 'Name of District', stage_col, 'Target']
feature_cols_res = [c for c in df_res.columns if c not in drop_cols]
X_res = df_res[feature_cols_res].apply(pd.to_numeric, errors='coerce')
y_res = df_res['Target']

# 2. Water Quality Target Formulation (BIS IS 10500:2012)
df_wq = df_wq_raw.copy()
ec_col = 'Electric Conductivity (μS/cm)'
tds_col = 'Total Dissolved Solids (mg/L)'
df_wq['TDS_Filled'] = pd.to_numeric(df_wq[tds_col], errors='coerce').fillna(pd.to_numeric(df_wq[ec_col], errors='coerce') * 0.64)

is_non_potable = (
    (pd.to_numeric(df_wq['Potential of Hydrogen (pH)'], errors='coerce') < 6.5) |
    (pd.to_numeric(df_wq['Potential of Hydrogen (pH)'], errors='coerce') > 8.5) |
    (df_wq['TDS_Filled'] > 500.0) |
    (pd.to_numeric(df_wq['Total Hardness (mgCaCO3/L)'], errors='coerce') > 200.0) |
    (pd.to_numeric(df_wq['Calcium (mg/L)'], errors='coerce') > 75.0) |
    (pd.to_numeric(df_wq['Magnesium (mg/L)'], errors='coerce') > 30.0) |
    (pd.to_numeric(df_wq['Chloride (mg/L)'], errors='coerce') > 250.0) |
    (pd.to_numeric(df_wq['Sulphate (mg/L)'], errors='coerce') > 200.0) |
    (pd.to_numeric(df_wq['Iron(mg/L)'], errors='coerce') > 1.0)
)
df_wq['Target'] = np.where(is_non_potable, 0, 1) # 0: Non-Potable, 1: Potable

feat_map = {
    'Potential of Hydrogen (pH)': 'pH',
    ec_col: 'Electrical_Conductivity',
    'TDS_Filled': 'TDS',
    'Total Hardness (mgCaCO3/L)': 'Total_Hardness',
    'Calcium (mg/L)': 'Calcium',
    'Magnesium (mg/L)': 'Magnesium',
    'Chloride (mg/L)': 'Chloride',
    'Sulphate (mg/L)': 'Sulphate',
    'Iron(mg/L)': 'Iron',
    'Bicarbonate (mg/L)': 'Bicarbonate',
    'Sodium (mg/L)': 'Sodium'
}
X_wq = df_wq[list(feat_map.keys())].rename(columns=feat_map).apply(pd.to_numeric, errors='coerce')
y_wq = df_wq['Target']

print(f'Water Quality class distribution:\n{y_wq.value_counts(normalize=True)}')

## 4. Train / Test Split & Model Comparison
We train and compare 6 classification models: Dummy Baseline, KNN, Decision Tree, Gaussian Naive Bayes, Random Forest, and XGBoost using Stratified 5-Fold Cross-Validation.

In [ ]:
# Stratified 80/20 train/test split
X_train_wq, X_test_wq, y_train_wq, y_test_wq = train_test_split(
    X_wq, y_wq, test_size=0.20, random_state=42, stratify=y_wq
)

models = {
    'Dummy Baseline': DummyClassifier(strategy='stratified', random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', clf)
    ])
    cv_scores = cross_validate(pipe, X_train_wq, y_train_wq, cv=cv, scoring='accuracy')
    pipe.fit(X_train_wq, y_train_wq)
    y_pred = pipe.predict(X_test_wq)
    
    results.append({
        'Model': name,
        'CV Accuracy': cv_scores['test_accuracy'].mean(),
        'Test Accuracy': accuracy_score(y_test_wq, y_pred),
        'Precision': precision_score(y_test_wq, y_pred, zero_division=0),
        'Recall': recall_score(y_test_wq, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test_wq, y_pred, zero_division=0),
        'Balanced Accuracy': balanced_accuracy_score(y_test_wq, y_pred)
    })

df_results = pd.DataFrame(results)
display(df_results)

## 5. Hyperparameter Tuning & Final Evaluation
Hyperparameter optimization for XGBoost and feature importance extraction.

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__max_depth': [3, 4, 6],
    'classifier__learning_rate': [0.03, 0.1, 0.2]
}
pipe_xgb = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])
grid_search = GridSearchCV(pipe_xgb, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_wq, y_train_wq)

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_wq)
print('Best Hyperparameters:', grid_search.best_params_)
print('\nTest Set Classification Report:\n', classification_report(y_test_wq, y_pred_best, target_names=['Non-Potable', 'Potable']))

# Confusion Matrix
cm = confusion_matrix(y_test_wq, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Potable', 'Potable'], yticklabels=['Non-Potable', 'Potable'])
plt.title('Water Quality Confusion Matrix (XGBoost)')
plt.xlabel('Predicted')
plt.ylabel('Actual BIS Ground Truth')
plt.show()

## 6. Live Prediction Function
Inference function ready for viva demonstration and integration with teammate modules.

In [ ]:
def test_sample_prediction(sample_dict):
    df_in = pd.DataFrame([sample_dict])
    pred = best_model.predict(df_in)[0]
    prob = best_model.predict_proba(df_in)[0]
    label = 'Potable' if pred == 1 else 'Non-Potable'
    print(f'Prediction: {label}')
    print(f'Confidence: {prob[pred]*100:.2f}%')
    return label

# Test with a sample
sample = {
    'pH': 7.6,
    'Electrical_Conductivity': 450.0,
    'TDS': 288.0,
    'Total_Hardness': 160.0,
    'Calcium': 45.0,
    'Magnesium': 18.0,
    'Chloride': 35.0,
    'Sulphate': 25.0,
    'Iron': 0.1,
    'Bicarbonate': 180.0,
    'Sodium': 32.0
}
test_sample_prediction(sample)